In [1]:
import os

random_data = os.urandom(4)
seed = int.from_bytes(random_data, byteorder="big")
print(f"Chosen Unbiased Seed: {seed}")

Chosen Unbiased Seed: 3013970066


In [2]:
import random
import numpy as np
import torch

def set_seed(seed: int):
    # ---- Python RNG ----
    random.seed(seed)

    # ---- NumPy RNG ----
    np.random.seed(seed)

    # ---- PyTorch CPU/GPU RNG ----
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Seed set to: {seed}")

set_seed(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

Seed set to: 3013970066


/gpu-data2/kfot/miniconda3/envs/myenv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import copy
import matplotlib.pyplot as plt

os.environ["CUDA_VISIBLE_DEVICES"]="3"

In [4]:
# Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
def flatten_tensors(tensors):
    return torch.cat([t.reshape(-1) for t in tensors])

def unflatten_like(flat, shapes):
    out = []
    idx = 0
    for s in shapes:
        numel = torch.tensor(s).prod().item()
        out.append(flat[idx:idx + numel].reshape(s))
        idx += numel
    return out

class MaskHandler:
    def __init__(self, model=None):
        self.param_to_mask = {}
        self.name_to_mask = {}

        if model is not None:
            self.register_model(model)

    def register_model(self, model):
        self.param_to_mask = {}
        self.name_to_mask = {}
        for name, p in model.named_parameters():
            mask = torch.ones_like(p)
            self.param_to_mask[id(p)] = mask
            self.name_to_mask[name] = mask

    def export(self):
        return {k: v.clone() for k, v in self.name_to_mask.items()}

    def load(self, model, mask_dict):
        self.param_to_mask = {}
        self.name_to_mask = {}

        for name, p in model.named_parameters():
            if name in mask_dict:
                m = mask_dict[name].to(p.device)
                self.param_to_mask[id(p)] = m
                self.name_to_mask[name] = m

        self.attach_model(model)
        
    def attach_model(self, model):
        for m in model.modules():
            if hasattr(m, "apply_mask"):
                m.mask_handler = self
        model.mask_handler = self

    def get(self, p):
        return self.param_to_mask.get(id(p), torch.ones_like(p))
    
    def set(self, p, m):
        self.param_to_mask[id(p)] = m

class Pruner:
    def __init__(self, model):
        self.model = model

        self.params = []
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.params.append((name, p))

        self.shapes = [p.shape for _, p in self.params]

        self.mask_handler = MaskHandler(self.model)
    
    def compute_scores_snip(self, dataloader, num_batches=10):
        self.model.train()

        # SNIP. We aggregate scores instead of gradients! 
        # Initial network -> untrained -> gradient over lots of batches ~= 0 (cancellation)
        # But, ~0 across all batches -> unimportant.
        # Variety across batches (and then cancellation) -> important. 
        scores = [torch.zeros_like(p) for _, p in self.params]

        for i, (x, y) in enumerate(dataloader):
            if i >= num_batches:
                break

            self.model.zero_grad()

            x, y = x.to(device), y.to(device)
            out = self.model(x)
            loss = F.cross_entropy(out, y)

            loss.backward()

            # SNIP criterion, note: |grad_c(c*w)| = |grad_w(w) * w|
            for j, (_, p) in enumerate(self.params):
                if p.grad is not None:
                    scores[j] += torch.abs(p.grad * p)

        return flatten_tensors(scores)
    
    def get_mask(self, scores, sparsity):
        k = int((1 - sparsity) * scores.numel())

        _, idx = torch.topk(scores, k=k, largest=True, sorted=False)

        mask = torch.zeros_like(scores, dtype=torch.bool)
        mask[idx] = True

        return mask
    
    def set_mask(self, flat_mask):
        masks = unflatten_like(flat_mask, self.shapes)

        with torch.no_grad():
            for (name, p), m in zip(self.params, masks):
                if "out" in name:
                    if len(m.shape) == 1:
                        safety_mask = torch.ones(m.shape, dtype=torch.bool, device=m.device)
                    else:
                        *prefix, last_dim = m.shape
                        idx = torch.randint(last_dim, tuple(prefix), device=m.device)
                        safety_mask = torch.nn.functional.one_hot(idx, num_classes=last_dim).to(torch.bool)
                    m = torch.maximum(m, safety_mask)
                self.mask_handler.set(p, m.to(p.device))
                self.mask_handler.name_to_mask[name] = m

        self.mask_handler.attach_model(self.model)

    def prune(self, dataloader, sparsity=0.5, num_batches=10, method="snip"):
        self.model.set_state("pruning")
        if method == "snip":
            scores = self.compute_scores_snip(dataloader, num_batches)
        else:
            scores = self.compute_scores_grasp(dataloader, num_batches)

        mask = self.get_mask(scores, sparsity)

        self.set_mask(mask)
        
        self.model.set_state("normal")

        return self.model.count_params()

class MLP(nn.Module):
    def __init__(self, input_size=784, hidden_size=256, num_classes=10, n_layers=5):
        super(MLP, self).__init__()
        self.n_layers = n_layers
        
        self.fc_weight, self.fc_bias = [], [] 
        self.fc_weight.append(nn.Parameter(torch.randn(hidden_size, input_size) * 0.01))
        self.fc_bias.append(nn.Parameter(torch.zeros(hidden_size)))
        for _ in range(n_layers - 1):
            self.fc_weight.append(nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01))
            self.fc_bias.append(nn.Parameter(torch.zeros(hidden_size)))

        self.fc_weight, self.fc_bias = nn.ParameterList(self.fc_weight), nn.ParameterList(self.fc_bias)

        self.out_weight = nn.Parameter(torch.randn(num_classes, hidden_size) * 0.01)
        self.out_bias = nn.Parameter(torch.zeros(num_classes))

        self.relu = nn.ReLU()

        self.mask_handler = None

        self.state = "normal" # does nothing for MLP
    
    def apply_mask(self, param):
        mask = self.mask_handler.get(param) if self.mask_handler is not None else torch.ones_like(param)
        return param * mask

    def set_state(self, state):
        self.state = state

    def count_params(self):
        total = sum(p.numel() for p in self.parameters())
        active = 0
        for name, p in self.named_parameters():
            mask = self.mask_handler.get(p) if self.mask_handler is not None else torch.ones_like(p)
            print(f"{name}: Initial params = {p.numel()}, Post-pruning active params = {mask.sum().item()}")
            active += mask.sum().item()
        return total, active

    def forward(self, x):
        x = x.view(-1, 784)

        for weight, bias in zip(self.fc_weight, self.fc_bias):
            x = F.linear(
                x,
                self.apply_mask(weight),
                self.apply_mask(bias),
            )
            x = self.relu(x)

        x = F.linear(
            x,
            self.apply_mask(self.out_weight),
            self.apply_mask(self.out_bias)
        )
        return x

class MorphologicalLayer(nn.Module):
    def __init__(self, input_size, output_size, bias=True, alpha=1, beta=0):
        super(MorphologicalLayer, self).__init__()
        self.bias = bias

        self.w = nn.Parameter(torch.randn(output_size, input_size)*alpha - beta)
        if bias:
            self.b = nn.Parameter(torch.randn(output_size))
            self.b2 = nn.Parameter(torch.randn(output_size))

        self.mask_handler = None

        self.state = "normal"
    
    def apply_mask(self, param, neg=False):
        mask = self.mask_handler.get(param) if self.mask_handler is not None else torch.ones_like(param)
        if neg:  # for min: masked values should be +inf
            return torch.where(mask.bool(), param, torch.full_like(param, float('inf'))), mask
        else:    # for max: masked values should be -inf
            return torch.where(mask.bool(), param, torch.full_like(param, float('-inf'))), mask

    def forward(self, x):
        x, valid = x
        x = x.unsqueeze(1) 
        valid = valid.unsqueeze(1)
        max_w, mask_w = self.apply_mask(self.w)
        min_w, _ = self.apply_mask(self.w, True)
        if self.bias:
            max_b, mask_b = self.apply_mask(self.b)
            min_b, mask_b2 = self.apply_mask(self.b2, True)
        x_max = x + max_w
        x_min = x + min_w
        if self.bias:
            x_max = torch.cat([x_max, max_b.view(1,-1,1).repeat(x.size(0),1,1)], dim=2)
            x_min = torch.cat([x_min, min_b.view(1,-1,1).repeat(x.size(0),1,1)], dim=2)
        valid_input = torch.minimum(valid, mask_w)
        # print("Valid input:", valid_input.size())
        valid_max = torch.cat([valid_input, mask_b.view(1,-1,1).repeat(x.size(0),1,1)], dim=2)
        valid_min = torch.cat([valid_input, mask_b2.view(1,-1,1).repeat(x.size(0),1,1)], dim=2)
        # print("Valid max/min:", valid_max.size(), valid_min.size())
        valid = torch.minimum(torch.max(valid_max, dim=2)[0], torch.max(valid_min, dim=2)[0]).bool()
        input_max = torch.where(valid_max.bool(), x_max, torch.full_like(x_max, float('-inf')))
        input_min = torch.where(valid_min.bool(), x_min, torch.full_like(x_min, float('inf')))
        x_max, _ = torch.max(input_max, dim=2)
        x_min, _ = torch.min(input_min, dim=2)
        x = x_max + x_min
        return (x, valid)
    
class LinAct(nn.Module):
    def __init__(self, size):
        super(LinAct, self).__init__()
        self.size = size        
        self.a = nn.Parameter(torch.randn(size) / 3.46)

        self.mask_handler = None
    
    def apply_mask(self, param):
        mask = self.mask_handler.get(param) if self.mask_handler is not None else torch.ones_like(param)
        return torch.where(mask.bool(), param, torch.zeros_like(param)), mask

    def forward(self, x):
        x, valid = x
        masked_a, mask = self.apply_mask(self.a)
        x = x * masked_a.view(1, -1).repeat(x.size(0), 1)
        return (x, valid)
    
class RMPM(nn.Module):
    def __init__(self, input_size=784, hidden_size=256, num_classes=10, n_layers=5):
        super(RMPM, self).__init__()
        self.n_layers = n_layers

        self.morph_layer = []
        self.morph_layer.append(nn.Sequential(MorphologicalLayer(input_size, hidden_size, alpha=0), LinAct(hidden_size)))
        for _ in range(n_layers - 1):
            self.morph_layer.append(nn.Sequential(MorphologicalLayer(hidden_size, hidden_size, alpha=1), LinAct(hidden_size)))
        
        self.morph_layer = nn.ModuleList(self.morph_layer)

        self.out = MorphologicalLayer(hidden_size, num_classes, alpha=1)

        self.mask_handler = None

        self.state = "normal"

    def count_params(self):
        total = sum(p.numel() for p in self.parameters())
        active = 0
        for name, p in self.named_parameters():
            mask = self.mask_handler.get(p) if self.mask_handler is not None else torch.ones_like(p)
            print(f"{name}: Initial params = {p.numel()}, Post-pruning active params = {mask.sum().item()}")
            active += mask.sum().item()
        return total, active

    def set_state(self, state):
        self.state = state
        for m in self.modules():
            if isinstance(m, MorphologicalLayer):
                m.state = state

    def forward(self, x):
        x = x.view(-1, 784)
        x = (x, torch.full_like(x, True))

        for layer in self.morph_layer:
            out, valid = layer(x)
            if out.size() == x[0].size():
                x = (x[0] + torch.where(valid, out, torch.zeros_like(out)), x[1])
            else:
                x = (out, valid)
        
        x, _ = self.out(x)

        return x

In [6]:
def train(model, criterion, optimizer, train_loader, val_loader, num_epochs=50, return_list=False):
    # Training and validation loop
    best_val_accuracy = 0.0
    best_model = None

    train_list = []
    val_list = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Used during development to ensure inactive parameters truly are inactive
            # with torch.no_grad():
            #     if model.mask_handler is not None:
            #         for name, p in model.named_parameters():
            #             mask = model.mask_handler.get(p)
            #             p.mul_(mask)

        # Validation phase
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total
        train_list.append(train_accuracy)
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = 100 * correct / total
        val_list.append(val_accuracy)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Train Accuracy: {train_accuracy:.2f}%, Validation Accuracy: {val_accuracy:.2f}%")

        # Save best model based on validation accuracy
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model = copy.deepcopy(model)
            if model.mask_handler is not None:
                mask_dict = model.mask_handler.export()

                best_model.mask_handler = MaskHandler()
                best_model.mask_handler.load(best_model, mask_dict)

    if return_list:
        return best_model, train_list, val_list
    else:
        return best_model

def test(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy on the test set: {accuracy:.2f}%')

In [7]:
# Load and preprocess MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

full_train_dataset = torchvision.datasets.MNIST(root='../data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='../data', train=False, transform=transform, download=True)

# Split train dataset into training and validation sets
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(seed))

In [8]:
# Data loaders
def make_loader(dataset, batch_size, shuffle, num_workers=0):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        worker_init_fn=seed_worker,
        generator=generator,
        pin_memory=True
    )

In [9]:
def run_experiment(model, train_dataset, val_dataset, test_dataset, pruning_ratio=0.5, num_batches=10, method="snip"):
    # Reset randomness
    set_seed(seed)
    train_loader = make_loader(train_dataset, batch_size=64, shuffle=True)
    val_loader = make_loader(val_dataset, batch_size=64, shuffle=False)
    test_loader = make_loader(test_dataset, batch_size=64, shuffle=False)

    model = model().to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model = train(model, criterion, optimizer, train_loader, val_loader, num_epochs=1)

    pruner = Pruner(model=model)
    params_initial, params_active = pruner.prune(train_loader, pruning_ratio, num_batches=num_batches, method=method)

    print(f"Initial number of parameters: {params_initial}")
    print(f"Total number of active parameters after pruning: {params_active}")

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model = train(model, criterion, optimizer, train_loader, val_loader)

    print(f"Total number of active parameters on testing: {model.count_params()[1]}")
    test(model, test_loader)

In [10]:
run_experiment(MLP, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.9875, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066
Epoch [1/1], Loss: 0.4877, Train Accuracy: 91.14%, Validation Accuracy: 90.39%
out_weight: Initial params = 2560, Post-pruning active params = 571
out_bias: Initial params = 10, Post-pruning active params = 10
fc_weight.0: Initial params = 200704, Post-pruning active params = 4009
fc_weight.1: Initial params = 65536, Post-pruning active params = 787
fc_weight.2: Initial params = 65536, Post-pruning active params = 137
fc_weight.3: Initial params = 65536, Post-pruning active params = 70
fc_weight.4: Initial params = 65536, Post-pruning active params = 213
fc_bias.0: Initial params = 256, Post-pruning active params = 0
fc_bias.1: Initial params = 256, Post-pruning active params = 10
fc_bias.2: Initial params = 256, Post-pruning active params = 11
fc_bias.3: Initial params = 256, Post-pruning active params = 9
fc_bias.4: Initial params = 256, Post-pruning active params = 14
Initial number of parameters: 466698
Total number of active parameters after pruning: 5841
E

In [11]:
run_experiment(MLP, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.99, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066
Epoch [1/1], Loss: 0.4877, Train Accuracy: 91.14%, Validation Accuracy: 90.39%
out_weight: Initial params = 2560, Post-pruning active params = 534
out_bias: Initial params = 10, Post-pruning active params = 10
fc_weight.0: Initial params = 200704, Post-pruning active params = 3151
fc_weight.1: Initial params = 65536, Post-pruning active params = 658
fc_weight.2: Initial params = 65536, Post-pruning active params = 95
fc_weight.3: Initial params = 65536, Post-pruning active params = 45
fc_weight.4: Initial params = 65536, Post-pruning active params = 145
fc_bias.0: Initial params = 256, Post-pruning active params = 0
fc_bias.1: Initial params = 256, Post-pruning active params = 9
fc_bias.2: Initial params = 256, Post-pruning active params = 9
fc_bias.3: Initial params = 256, Post-pruning active params = 8
fc_bias.4: Initial params = 256, Post-pruning active params = 10
Initial number of parameters: 466698
Total number of active parameters after pruning: 4674
Epoc

In [12]:
run_experiment(MLP, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.9925, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066
Epoch [1/1], Loss: 0.4877, Train Accuracy: 91.14%, Validation Accuracy: 90.39%
out_weight: Initial params = 2560, Post-pruning active params = 500
out_bias: Initial params = 10, Post-pruning active params = 10
fc_weight.0: Initial params = 200704, Post-pruning active params = 2287
fc_weight.1: Initial params = 65536, Post-pruning active params = 503
fc_weight.2: Initial params = 65536, Post-pruning active params = 62
fc_weight.3: Initial params = 65536, Post-pruning active params = 29
fc_weight.4: Initial params = 65536, Post-pruning active params = 93
fc_bias.0: Initial params = 256, Post-pruning active params = 0
fc_bias.1: Initial params = 256, Post-pruning active params = 4
fc_bias.2: Initial params = 256, Post-pruning active params = 7
fc_bias.3: Initial params = 256, Post-pruning active params = 7
fc_bias.4: Initial params = 256, Post-pruning active params = 6
Initial number of parameters: 466698
Total number of active parameters after pruning: 3508
Epoch 

In [13]:
run_experiment(MLP, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.995, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066
Epoch [1/1], Loss: 0.4877, Train Accuracy: 91.14%, Validation Accuracy: 90.39%
out_weight: Initial params = 2560, Post-pruning active params = 441
out_bias: Initial params = 10, Post-pruning active params = 10
fc_weight.0: Initial params = 200704, Post-pruning active params = 1438
fc_weight.1: Initial params = 65536, Post-pruning active params = 356
fc_weight.2: Initial params = 65536, Post-pruning active params = 34
fc_weight.3: Initial params = 65536, Post-pruning active params = 11
fc_weight.4: Initial params = 65536, Post-pruning active params = 38
fc_bias.0: Initial params = 256, Post-pruning active params = 0
fc_bias.1: Initial params = 256, Post-pruning active params = 3
fc_bias.2: Initial params = 256, Post-pruning active params = 5
fc_bias.3: Initial params = 256, Post-pruning active params = 2
fc_bias.4: Initial params = 256, Post-pruning active params = 4
Initial number of parameters: 466698
Total number of active parameters after pruning: 2342
Epoch 

In [14]:
run_experiment(RMPM, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.9875, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066
Epoch [1/1], Loss: 0.5034, Train Accuracy: 88.87%, Validation Accuracy: 88.34%
morph_layer.0.0.w: Initial params = 200704, Post-pruning active params = 2094
morph_layer.0.0.b: Initial params = 256, Post-pruning active params = 44
morph_layer.0.0.b2: Initial params = 256, Post-pruning active params = 34
morph_layer.0.1.a: Initial params = 256, Post-pruning active params = 236
morph_layer.1.0.w: Initial params = 65536, Post-pruning active params = 958
morph_layer.1.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.1.0.b2: Initial params = 256, Post-pruning active params = 0
morph_layer.1.1.a: Initial params = 256, Post-pruning active params = 191
morph_layer.2.0.w: Initial params = 65536, Post-pruning active params = 985
morph_layer.2.0.b: Initial params = 256, Post-pruning active params = 2
morph_layer.2.0.b2: Initial params = 256, Post-pruning active params = 1
morph_layer.2.1.a: Initial params = 256, Post-pruning active params = 157
morph_la

In [15]:
run_experiment(RMPM, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.99, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066


Epoch [1/1], Loss: 0.5034, Train Accuracy: 88.87%, Validation Accuracy: 88.34%
morph_layer.0.0.w: Initial params = 200704, Post-pruning active params = 1462
morph_layer.0.0.b: Initial params = 256, Post-pruning active params = 43
morph_layer.0.0.b2: Initial params = 256, Post-pruning active params = 34
morph_layer.0.1.a: Initial params = 256, Post-pruning active params = 225
morph_layer.1.0.w: Initial params = 65536, Post-pruning active params = 859
morph_layer.1.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.1.0.b2: Initial params = 256, Post-pruning active params = 0
morph_layer.1.1.a: Initial params = 256, Post-pruning active params = 170
morph_layer.2.0.w: Initial params = 65536, Post-pruning active params = 822
morph_layer.2.0.b: Initial params = 256, Post-pruning active params = 2
morph_layer.2.0.b2: Initial params = 256, Post-pruning active params = 1
morph_layer.2.1.a: Initial params = 256, Post-pruning active params = 130
morph_layer.3.0.w: Initial param

In [16]:
run_experiment(RMPM, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.9925, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066


Epoch [1/1], Loss: 0.5034, Train Accuracy: 88.87%, Validation Accuracy: 88.34%
morph_layer.0.0.w: Initial params = 200704, Post-pruning active params = 901
morph_layer.0.0.b: Initial params = 256, Post-pruning active params = 40
morph_layer.0.0.b2: Initial params = 256, Post-pruning active params = 31
morph_layer.0.1.a: Initial params = 256, Post-pruning active params = 201
morph_layer.1.0.w: Initial params = 65536, Post-pruning active params = 724
morph_layer.1.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.1.0.b2: Initial params = 256, Post-pruning active params = 0
morph_layer.1.1.a: Initial params = 256, Post-pruning active params = 142
morph_layer.2.0.w: Initial params = 65536, Post-pruning active params = 652
morph_layer.2.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.2.0.b2: Initial params = 256, Post-pruning active params = 1
morph_layer.2.1.a: Initial params = 256, Post-pruning active params = 113
morph_layer.3.0.w: Initial params

In [17]:
run_experiment(RMPM, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.995, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066


Epoch [1/1], Loss: 0.5034, Train Accuracy: 88.87%, Validation Accuracy: 88.34%
morph_layer.0.0.w: Initial params = 200704, Post-pruning active params = 430
morph_layer.0.0.b: Initial params = 256, Post-pruning active params = 30
morph_layer.0.0.b2: Initial params = 256, Post-pruning active params = 25
morph_layer.0.1.a: Initial params = 256, Post-pruning active params = 155
morph_layer.1.0.w: Initial params = 65536, Post-pruning active params = 569
morph_layer.1.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.1.0.b2: Initial params = 256, Post-pruning active params = 0
morph_layer.1.1.a: Initial params = 256, Post-pruning active params = 102
morph_layer.2.0.w: Initial params = 65536, Post-pruning active params = 459
morph_layer.2.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.2.0.b2: Initial params = 256, Post-pruning active params = 0
morph_layer.2.1.a: Initial params = 256, Post-pruning active params = 79
morph_layer.3.0.w: Initial params 

In [18]:
# Load and preprocess MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

full_train_dataset = torchvision.datasets.FashionMNIST(root='../data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.FashionMNIST(root='../data', train=False, transform=transform, download=True)

# Split train dataset into training and validation sets
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(seed))

In [19]:
run_experiment(MLP, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.9875, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066


Epoch [1/1], Loss: 0.7398, Train Accuracy: 79.45%, Validation Accuracy: 79.33%
out_weight: Initial params = 2560, Post-pruning active params = 841
out_bias: Initial params = 10, Post-pruning active params = 10
fc_weight.0: Initial params = 200704, Post-pruning active params = 2459
fc_weight.1: Initial params = 65536, Post-pruning active params = 862
fc_weight.2: Initial params = 65536, Post-pruning active params = 271
fc_weight.3: Initial params = 65536, Post-pruning active params = 402
fc_weight.4: Initial params = 65536, Post-pruning active params = 993
fc_bias.0: Initial params = 256, Post-pruning active params = 1
fc_bias.1: Initial params = 256, Post-pruning active params = 0
fc_bias.2: Initial params = 256, Post-pruning active params = 0
fc_bias.3: Initial params = 256, Post-pruning active params = 0
fc_bias.4: Initial params = 256, Post-pruning active params = 0
Initial number of parameters: 466698
Total number of active parameters after pruning: 5839
Epoch [1/50], Loss: 1.0451,

In [20]:
run_experiment(MLP, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.99, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066
Epoch [1/1], Loss: 0.7398, Train Accuracy: 79.45%, Validation Accuracy: 79.33%
out_weight: Initial params = 2560, Post-pruning active params = 816
out_bias: Initial params = 10, Post-pruning active params = 10
fc_weight.0: Initial params = 200704, Post-pruning active params = 1949
fc_weight.1: Initial params = 65536, Post-pruning active params = 689
fc_weight.2: Initial params = 65536, Post-pruning active params = 168
fc_weight.3: Initial params = 65536, Post-pruning active params = 274
fc_weight.4: Initial params = 65536, Post-pruning active params = 766
fc_bias.0: Initial params = 256, Post-pruning active params = 0
fc_bias.1: Initial params = 256, Post-pruning active params = 0
fc_bias.2: Initial params = 256, Post-pruning active params = 0
fc_bias.3: Initial params = 256, Post-pruning active params = 0
fc_bias.4: Initial params = 256, Post-pruning active params = 0
Initial number of parameters: 466698
Total number of active parameters after pruning: 4672


Epoch [1/50], Loss: 0.9821, Train Accuracy: 71.37%, Validation Accuracy: 70.95%
Epoch [2/50], Loss: 0.8217, Train Accuracy: 77.46%, Validation Accuracy: 76.83%
Epoch [3/50], Loss: 0.6128, Train Accuracy: 79.40%, Validation Accuracy: 78.71%
Epoch [4/50], Loss: 0.4921, Train Accuracy: 81.17%, Validation Accuracy: 80.31%
Epoch [5/50], Loss: 0.4680, Train Accuracy: 81.67%, Validation Accuracy: 80.94%
Epoch [6/50], Loss: 0.6004, Train Accuracy: 82.09%, Validation Accuracy: 82.12%
Epoch [7/50], Loss: 0.5116, Train Accuracy: 82.84%, Validation Accuracy: 82.67%
Epoch [8/50], Loss: 0.4255, Train Accuracy: 83.28%, Validation Accuracy: 83.11%
Epoch [9/50], Loss: 0.3764, Train Accuracy: 83.36%, Validation Accuracy: 82.67%
Epoch [10/50], Loss: 0.2403, Train Accuracy: 83.74%, Validation Accuracy: 83.33%
Epoch [11/50], Loss: 0.3043, Train Accuracy: 83.97%, Validation Accuracy: 83.58%
Epoch [12/50], Loss: 0.3839, Train Accuracy: 83.93%, Validation Accuracy: 83.31%
Epoch [13/50], Loss: 0.3881, Train Ac

In [21]:
run_experiment(MLP, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.9925, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066


Epoch [1/1], Loss: 0.7398, Train Accuracy: 79.45%, Validation Accuracy: 79.33%
out_weight: Initial params = 2560, Post-pruning active params = 788
out_bias: Initial params = 10, Post-pruning active params = 10
fc_weight.0: Initial params = 200704, Post-pruning active params = 1451
fc_weight.1: Initial params = 65536, Post-pruning active params = 514
fc_weight.2: Initial params = 65536, Post-pruning active params = 82
fc_weight.3: Initial params = 65536, Post-pruning active params = 150
fc_weight.4: Initial params = 65536, Post-pruning active params = 511
fc_bias.0: Initial params = 256, Post-pruning active params = 0
fc_bias.1: Initial params = 256, Post-pruning active params = 0
fc_bias.2: Initial params = 256, Post-pruning active params = 0
fc_bias.3: Initial params = 256, Post-pruning active params = 0
fc_bias.4: Initial params = 256, Post-pruning active params = 0
Initial number of parameters: 466698
Total number of active parameters after pruning: 3506
Epoch [1/50], Loss: 1.1807, 

In [22]:
run_experiment(MLP, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.995, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066


Epoch [1/1], Loss: 0.7398, Train Accuracy: 79.45%, Validation Accuracy: 79.33%
out_weight: Initial params = 2560, Post-pruning active params = 751
out_bias: Initial params = 10, Post-pruning active params = 10
fc_weight.0: Initial params = 200704, Post-pruning active params = 925
fc_weight.1: Initial params = 65536, Post-pruning active params = 298
fc_weight.2: Initial params = 65536, Post-pruning active params = 35
fc_weight.3: Initial params = 65536, Post-pruning active params = 51
fc_weight.4: Initial params = 65536, Post-pruning active params = 270
fc_bias.0: Initial params = 256, Post-pruning active params = 0
fc_bias.1: Initial params = 256, Post-pruning active params = 0
fc_bias.2: Initial params = 256, Post-pruning active params = 0
fc_bias.3: Initial params = 256, Post-pruning active params = 0
fc_bias.4: Initial params = 256, Post-pruning active params = 0
Initial number of parameters: 466698
Total number of active parameters after pruning: 2340
Epoch [1/50], Loss: 1.3648, Tr

In [23]:
run_experiment(RMPM, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.9875, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066


Epoch [1/1], Loss: 0.6990, Train Accuracy: 80.25%, Validation Accuracy: 79.33%
morph_layer.0.0.w: Initial params = 200704, Post-pruning active params = 2405
morph_layer.0.0.b: Initial params = 256, Post-pruning active params = 51
morph_layer.0.0.b2: Initial params = 256, Post-pruning active params = 34
morph_layer.0.1.a: Initial params = 256, Post-pruning active params = 239
morph_layer.1.0.w: Initial params = 65536, Post-pruning active params = 961
morph_layer.1.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.1.0.b2: Initial params = 256, Post-pruning active params = 0
morph_layer.1.1.a: Initial params = 256, Post-pruning active params = 199
morph_layer.2.0.w: Initial params = 65536, Post-pruning active params = 835
morph_layer.2.0.b: Initial params = 256, Post-pruning active params = 2
morph_layer.2.0.b2: Initial params = 256, Post-pruning active params = 1
morph_layer.2.1.a: Initial params = 256, Post-pruning active params = 143
morph_layer.3.0.w: Initial param

In [24]:
run_experiment(RMPM, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.99, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066


Epoch [1/1], Loss: 0.6990, Train Accuracy: 80.25%, Validation Accuracy: 79.33%
morph_layer.0.0.w: Initial params = 200704, Post-pruning active params = 1562
morph_layer.0.0.b: Initial params = 256, Post-pruning active params = 49
morph_layer.0.0.b2: Initial params = 256, Post-pruning active params = 34
morph_layer.0.1.a: Initial params = 256, Post-pruning active params = 233
morph_layer.1.0.w: Initial params = 65536, Post-pruning active params = 895
morph_layer.1.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.1.0.b2: Initial params = 256, Post-pruning active params = 0
morph_layer.1.1.a: Initial params = 256, Post-pruning active params = 186
morph_layer.2.0.w: Initial params = 65536, Post-pruning active params = 751
morph_layer.2.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.2.0.b2: Initial params = 256, Post-pruning active params = 1
morph_layer.2.1.a: Initial params = 256, Post-pruning active params = 132
morph_layer.3.0.w: Initial param

In [25]:
run_experiment(RMPM, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.9925, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066


Epoch [1/1], Loss: 0.6990, Train Accuracy: 80.25%, Validation Accuracy: 79.33%
morph_layer.0.0.w: Initial params = 200704, Post-pruning active params = 872
morph_layer.0.0.b: Initial params = 256, Post-pruning active params = 45
morph_layer.0.0.b2: Initial params = 256, Post-pruning active params = 33
morph_layer.0.1.a: Initial params = 256, Post-pruning active params = 209
morph_layer.1.0.w: Initial params = 65536, Post-pruning active params = 780
morph_layer.1.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.1.0.b2: Initial params = 256, Post-pruning active params = 0
morph_layer.1.1.a: Initial params = 256, Post-pruning active params = 162
morph_layer.2.0.w: Initial params = 65536, Post-pruning active params = 625
morph_layer.2.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.2.0.b2: Initial params = 256, Post-pruning active params = 1
morph_layer.2.1.a: Initial params = 256, Post-pruning active params = 112
morph_layer.3.0.w: Initial params

In [26]:
run_experiment(RMPM, 
               train_dataset, 
               val_dataset, 
               test_dataset, 
               pruning_ratio=0.995, 
               num_batches=10, 
               method="snip"
)

Seed set to: 3013970066


Epoch [1/1], Loss: 0.6990, Train Accuracy: 80.25%, Validation Accuracy: 79.33%
morph_layer.0.0.w: Initial params = 200704, Post-pruning active params = 382
morph_layer.0.0.b: Initial params = 256, Post-pruning active params = 37
morph_layer.0.0.b2: Initial params = 256, Post-pruning active params = 25
morph_layer.0.1.a: Initial params = 256, Post-pruning active params = 157
morph_layer.1.0.w: Initial params = 65536, Post-pruning active params = 607
morph_layer.1.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.1.0.b2: Initial params = 256, Post-pruning active params = 0
morph_layer.1.1.a: Initial params = 256, Post-pruning active params = 106
morph_layer.2.0.w: Initial params = 65536, Post-pruning active params = 467
morph_layer.2.0.b: Initial params = 256, Post-pruning active params = 1
morph_layer.2.0.b2: Initial params = 256, Post-pruning active params = 1
morph_layer.2.1.a: Initial params = 256, Post-pruning active params = 85
morph_layer.3.0.w: Initial params 